# Workspace Coverage Analysis

Loads the most recent `workspace_coverage` run and visualises planned vs measured vs FK across the reachable envelope. Volcaniarm is a 2-DOF planar arm so X is fixed mechanical bias (not an error metric); accuracy is measured in the Y-Z plane only.

Two complementary views:
1. **Vector residual** in the Y-Z plane (`dy, dz, dr_yz`) — biased by URDF mount placeholders, but variance/spread across goals is informative.
2. **Scalar Euclidean distance** `|EE − base|` from FK and from tag — the bias *shifts* across goals if the URDF mounts are wrong, so the spread of `d_tag − d_fk` across goals is itself diagnostic of URDF mount accuracy.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from volcaniarm_calibration.analysis import (
    align_fk_to_tag, latest_run, list_runs, load_run,
)

TEST_NAME = 'workspace_coverage'
RUN_DIR = latest_run(TEST_NAME)
run = load_run(RUN_DIR)
config, tag, fk = run['config'], run['tag'], run['fk']
print(f'run_id:  {config.get("run_id")}')
print(f'status:  {config.get("status")}  failure_reason: {config.get("failure_reason")}')
print(f'goals:   {config["goals"]}')
print(f'samples per visit: {config["samples_per_visit"]}')
print(f'tag rows: {len(tag)}, fk rows: {len(fk)}')

## Goals + measurements (Y-Z plane)

Top-down workspace view. Each goal is one star; FK predictions cluster on the goals (they're deterministic in joint angles); tag measurements are camera-derived in `apriltag_marker_base` frame and offset by the URDF placeholder.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 7))

goals = np.array(config['goals'])
ax.scatter(goals[:, 0], goals[:, 1], marker='*', s=300, c='red',
           label='goals', zorder=5)
ax.scatter(fk['fk_y'], fk['fk_z'], marker='o', s=120, facecolors='none',
           edgecolors='orange', linewidths=2, label='FK', zorder=4)
target = tag[tag['phase'] == 'target']
ax.scatter(target['y'], target['z'], marker='x', s=80, c='steelblue',
           label='tag-measured', zorder=3)
for i, (gy, gz) in enumerate(goals, start=1):
    ax.annotate(f'g{i}', (gy, gz), textcoords='offset points',
                xytext=(6, 6), fontsize=8, color='darkred')
ax.set_xlabel('y [m]')
ax.set_ylabel('z [m]')
ax.set_title(f'workspace y-z view  ({config["run_id"]})')
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')
ax.legend()
fig.tight_layout()

## Per-goal Euclidean distance: |EE − base| from FK vs tag

Per goal, the distance from base origin to EE origin (FK) vs the distance between AprilTag centres (tag). The difference per goal carries the URDF mount placeholder; if the URDF is right, the difference should be **constant across goals**. Variation in `d_diff` across goals is a signature of wrong URDF mount offsets.

In [ ]:
tag_target = tag[tag['phase'] == 'target'].copy()
tag_target['d_tag_m'] = np.sqrt(
    tag_target['x'] ** 2 + tag_target['y'] ** 2 + tag_target['z'] ** 2)
fk_distances = fk.copy()
fk_distances['d_fk_m'] = np.sqrt(
    fk_distances['fk_x'] ** 2 + fk_distances['fk_y'] ** 2 + fk_distances['fk_z'] ** 2)

tag_target['target_idx'] = tag_target['target_idx'].astype(int)
fk_distances['target_idx'] = fk_distances['target_idx'].astype(int)

# One row per (cycle, target_idx, sample_idx) with both distances
merged_d = tag_target.merge(
    fk_distances[['cycle', 'target_idx', 'd_fk_m']],
    on=['cycle', 'target_idx'], how='left')
merged_d['d_diff_m'] = merged_d['d_tag_m'] - merged_d['d_fk_m']

# Per-goal aggregate (mean across samples within the goal visit)
by_goal = merged_d.groupby('target_idx').agg(
    d_tag_mean=('d_tag_m', 'mean'),
    d_tag_std=('d_tag_m', 'std'),
    d_fk=('d_fk_m', 'first'),
    d_diff=('d_diff_m', 'mean'),
).reset_index()
by_goal['d_diff_mm'] = by_goal['d_diff'] * 1000
by_goal['d_tag_std_mm'] = by_goal['d_tag_std'] * 1000
print(by_goal[['target_idx', 'd_fk', 'd_tag_mean', 'd_tag_std_mm',
               'd_diff_mm']].to_string(index=False))
print(f'\nd_diff across goals: mean = {by_goal["d_diff_mm"].mean():+.2f} mm, '
      f'std = {by_goal["d_diff_mm"].std():.2f} mm')
print('  (std should be small if URDF mounts are accurate; large std = URDF placeholder error varying with EE pose)')

## Per-goal residual table (Y-Z components, sorted by magnitude)

`align_fk_to_tag` joins on `(cycle, target_idx)`; here we average each goal's samples and rank by `dr_yz`. Outliers point at goals where measurement was bad (e.g., tag partially occluded, EE at edge of workspace).

In [ ]:
merged = align_fk_to_tag(tag, fk)
merged['dr_yz'] = (merged['dy'] ** 2 + merged['dz'] ** 2) ** 0.5
by_goal_resid = merged.groupby('target_idx').agg(
    n_samples=('sample_idx', 'count'),
    dy_mean=('dy', 'mean'),
    dz_mean=('dz', 'mean'),
    dr_yz_mean=('dr_yz', 'mean'),
    dr_yz_std=('dr_yz', 'std'),
).reset_index()
by_goal_resid['dr_yz_mean_mm'] = by_goal_resid['dr_yz_mean'] * 1000
by_goal_resid['dr_yz_std_mm'] = by_goal_resid['dr_yz_std'] * 1000
by_goal_resid = by_goal_resid.sort_values('dr_yz_mean_mm', ascending=False)
print(by_goal_resid[['target_idx', 'n_samples', 'dy_mean', 'dz_mean',
                     'dr_yz_mean_mm', 'dr_yz_std_mm']].to_string(index=False))

## 2D heatmap of d_diff over (y, z)

Diagnostic for URDF mount placeholder accuracy: if the URDF is correct, the colour bar should be roughly uniform. If colours vary spatially, the URDF mount offsets are wrong in a pose-dependent way.

In [ ]:
if len(by_goal) >= 3:
    # tricontourf needs at least 3 (y, z) points to triangulate.
    goals_arr = np.array(config['goals'])
    ys = goals_arr[by_goal['target_idx'].values - 1, 0]
    zs = goals_arr[by_goal['target_idx'].values - 1, 1]
    fig, ax = plt.subplots(1, 1, figsize=(8, 7))
    tcf = ax.tricontourf(ys, zs, by_goal['d_diff_mm'].values,
                         levels=14, cmap='RdBu_r')
    ax.scatter(ys, zs, c='black', s=30, zorder=5)
    for ti, y, z, val in zip(by_goal['target_idx'].values, ys, zs,
                              by_goal['d_diff_mm'].values):
        ax.annotate(f'g{ti}\n{val:+.0f}', (y, z),
                    textcoords='offset points', xytext=(6, 6), fontsize=8)
    cbar = plt.colorbar(tcf, ax=ax)
    cbar.set_label('d_tag − d_fk [mm]')
    ax.set_xlabel('y [m]')
    ax.set_ylabel('z [m]')
    ax.set_title('|EE−base| difference (tag − FK) over workspace')
    ax.set_aspect('equal')
    fig.tight_layout()
else:
    print(f'only {len(by_goal)} goals; need at least 3 for the heatmap')